# Multi-Agent Sales Workflow with watsonx.governance

This notebook demonstrates the multi-agent sales workflow with integrated watsonx.governance evaluation.

## Governance Metrics

- **Email Generation Quality:** Faithfulness and professionalism of generated emails
- **Overall Workflow Quality:** Completeness and accuracy of recommendations

## Setup

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import uuid

load_dotenv()

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Initialize Multi-Agent System

In [2]:
from supervisory_agent import SupervisoryAgent

# MODEL OPTIONS (if you encounter connection errors, try alternatives):
# - "meta-llama/llama-4-maverick-17b-128e-instruct-fp8"  (Primary - Testing Llama 4)
# - "meta-llama/llama-3-3-70b-instruct"  (Alternative 1 - Best performance)
# - "meta-llama/llama-3-1-70b-gptq"      (Alternative 2)
# - "meta-llama/llama-3-1-8b"            (Alternative 3 - Faster, lower cost)
# - "ibm/granite-3-3-8b-instruct-np"     (Alternative 4 - IBM model)

supervisor = SupervisoryAgent(
    model_id="meta-llama/llama-4-maverick-17b-128e-instruct-fp8",  # Testing Llama 4
    url=os.getenv("WATSONX_URL", "https://us-south.ml.cloud.ibm.com"),
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID"),
    contract_vector_store_path="./contract_vector_store",
    crm_file_path="docs/Confluent Sales Cloud Infor.xlsx"
)

print("[SUCCESS] Multi-agent system initialized")

[SUCCESS] Multi-agent system initialized


## Initialize watsonx.governance Evaluator

In [3]:
from ibm_watsonx_gov.evaluators.metrics_evaluator import MetricsEvaluator
from ibm_watsonx_gov.metrics import FaithfulnessMetric
from ibm_watsonx_gov.config import GenAIConfiguration
from ibm_watsonx_gov.entities.foundation_model import WxAIFoundationModel
from ibm_watsonx_gov.entities.llm_judge import LLMJudge

PROJECT_ID = os.getenv("WATSONX_PROJECT_ID")
REGION = "us-south"  

# Use Llama 3.3 70B for LLM judge (more stable than Llama 4 FP8 for evaluation)
# Keep Llama 4 for agents, but use Llama 3.3 for judging
llm_judge = LLMJudge(
    model=WxAIFoundationModel(
        model_id="meta-llama/llama-3-3-70b-instruct",
        project_id=PROJECT_ID,
        region=REGION
    )
)

evaluator = MetricsEvaluator(
    project_id=PROJECT_ID,
    region=REGION
)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/ibm_watsonx_gov/tools/utils/package_utils.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/ibm_watsonx_gov/tools/utils/environment.py:55: UserWarning: Since WATSONX_REGION is not provided in the environment variable, the Dallas region will be used as the default.
  warnings.warn(
[2026-04-21 11:19:39,770]-[ibm_watsonx_gov.evaluators.agentic_evaluator]-[ WARNING ]-[Line 125] ~~> No module named 'ibm_agent_analytics'


## Create Test Queries

In [4]:
# Using only query 0 for focused governance evaluation
test_queries = [
    "I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps"
]

query_df = pd.DataFrame({"input_text": test_queries})
query_df["message_id"] = [str(uuid.uuid4()) for _ in range(len(query_df))]
query_df["partner_name"] = "Confluent"

query_df

,input_text,message_id,partner_name
0,"I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps",4aacbf98-d9b8-4bed-8363-cb7dede06a3f,Confluent


## Run Workflow with Governance Evaluation

In [5]:
batch_results = []
agent_results = []

for idx, row in query_df.iterrows():
    print("="*80)
    print(f"Processing Query {idx+1}/{len(query_df)}")
    print("="*80)
    print(f"Query: {row['input_text'][:100]}...")
    print()
    
    try:
        result = supervisor.run(
            seller_query=row["input_text"],
            contract_file_path=None,
            partner_name=row["partner_name"]
        )
        
        action_rec = result.get("action_recommendation", {})
        draft_email = action_rec.get("draft_email", "")
        risk_level = action_rec.get("risk_assessment", {}).get("risk_level", "Unknown")
        
        # Store agent result
        agent_results.append({
            "message_id": row["message_id"],
            "input_text": row["input_text"],
            "generated_text": draft_email,
            "risk_level": risk_level,
            "context": str(result.get("contract_summary", {}))[:500] + " | " + str(result.get("partner_profile", {}))[:500]
        })
        
        print(f"[SUCCESS] Workflow completed")
        print(f"Risk Level: {risk_level}")
        print(f"Email Length: {len(draft_email)} characters")
        
    except Exception as e:
        print(f"[ERROR] {str(e)}")
        agent_results.append({
            "message_id": row["message_id"],
            "input_text": row["input_text"],
            "generated_text": f"Error: {str(e)}",
            "risk_level": "Error",
            "context": ""
        })
    
    print("\n")

agent_df = pd.DataFrame(agent_results)

print("="*80)
print("BATCH PROCESSING COMPLETE")
print("="*80)

Processing Query 1/1
Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what co...


SUPERVISORY AGENT - Workflow Initialization
Seller Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps

Workflow Type: renewal_expiration_awareness
Required Agents: contract, action
Partner Name: Confluent

EXECUTING CONTRACT AGENT
Preloading contract portfolio for partner: Confluent
Contract scope: all files in docs/ beginning with Confluent_IBM
✓ Using cached data for: Confluent_IBM-1.30.2024.docx
✓ Using cached metadata for: Confluent_IBM-1.30.2024.docx
✓ Using cached structured data for: Confluent_IBM-1.30.2024.docx
✓ Using cached data for: Confluent_IBM-1.30.2025.docx
✓ Using cached metadata for: Confluent_IBM-1.30.2025

## Extract Next Steps and High Urgency Actions for Evaluation

In [6]:
print("="*80)
print("EXTRACTING NEXT STEPS FOR GOVERNANCE EVALUATION")
print("="*80)

# Extract next steps and high urgency actions from workflow results
next_steps_results = []

for idx, row in query_df.iterrows():
    try:
        result = supervisor.run(
            seller_query=row["input_text"],
            contract_file_path=None,
            partner_name=row["partner_name"]
        )
        
        action_rec = result.get("action_recommendation", {})
        ranked_next_steps = action_rec.get("ranked_next_steps", [])
        detailed_actions = action_rec.get("detailed_actions", [])
        
        # Get high urgency next step (top priority)
        high_urgency_step = ranked_next_steps[0] if ranked_next_steps else "No action identified"
        
        # Get all next steps as a single text
        all_next_steps = " | ".join(ranked_next_steps[:5]) if ranked_next_steps else "No next steps identified"
        
        # Extract context from contract and CRM data
        context_parts = []
        if detailed_actions:
            top_action = detailed_actions[0]
            context_parts.append(f"Contract: {top_action.get('contract', 'Unknown')}")
            context_parts.append(f"Urgency: {top_action.get('urgency_level', 'Unknown')}")
            context_parts.append(f"Priority: {top_action.get('priority', 'Unknown')}")
            if top_action.get('recipient_info'):
                recipient = top_action['recipient_info']
                context_parts.append(f"Recipient: {recipient.get('name', recipient.get('role', 'Unknown'))}")
        
        context_text = " | ".join(context_parts) if context_parts else str(result.get("contract_summary", {}))[:500]
        
        next_steps_results.append({
            "message_id": row["message_id"],
            "input_text": row["input_text"],
            "high_urgency_step": high_urgency_step,
            "all_next_steps": all_next_steps,
            "context": context_text
        })
        
        print(f"[SUCCESS] Extracted next steps for query {idx+1}")
        
    except Exception as e:
        print(f"[ERROR] Failed to extract next steps for query {idx+1}: {str(e)}")
        next_steps_results.append({
            "message_id": row["message_id"],
            "input_text": row["input_text"],
            "high_urgency_step": f"Error: {str(e)}",
            "all_next_steps": f"Error: {str(e)}",
            "context": ""
        })

next_steps_df = pd.DataFrame(next_steps_results)
print(f"\n[SUCCESS] Extracted next steps for {len(next_steps_df)} queries")

EXTRACTING NEXT STEPS FOR GOVERNANCE EVALUATION

SUPERVISORY AGENT - Workflow Initialization
Seller Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps

Workflow Type: renewal_expiration_awareness
Required Agents: contract, action
Partner Name: Confluent

EXECUTING CONTRACT AGENT
Preloading contract portfolio for partner: Confluent
Contract scope: all files in docs/ beginning with Confluent_IBM
✓ Using cached data for: Confluent_IBM-1.30.2024.docx
✓ Using cached metadata for: Confluent_IBM-1.30.2024.docx
✓ Using cached structured data for: Confluent_IBM-1.30.2024.docx
✓ Using cached data for: Confluent_IBM-1.30.2025.docx
✓ Using cached metadata for: Confluent_IBM-1.30.2025.docx
✓ Using cached structured data for: Confluent_IBM-1.30.2025.docx
✓ Using cached

## Evaluate Next Steps Quality with Governance

In [7]:
print("\n" + "="*80)
print("EVALUATING NEXT STEPS QUALITY WITH WATSONX.GOVERNANCE")
print("="*80)

# Filter out error results
valid_next_steps = next_steps_df[~next_steps_df["high_urgency_step"].str.contains("Error", na=False)].copy()

if not valid_next_steps.empty:
    # Prepare evaluation data for high urgency steps
    eval_urgency_data = valid_next_steps[["input_text", "context"]].copy()
    eval_urgency_data["generated_text"] = valid_next_steps["high_urgency_step"]
    
    # Create faithfulness metric for next steps
    config_urgency = GenAIConfiguration(
        input_fields=["input_text"],
        context_fields=["context"],
        output_fields=["generated_text"]
    )
    
    faithfulness_urgency = FaithfulnessMetric(
        llm_judge=llm_judge,
        configuration=config_urgency
    )
    
    try:
        print(f"\nEvaluating {len(eval_urgency_data)} high urgency next steps...")
        print(f"\n[DEBUG] Next steps data shape: {eval_urgency_data.shape}")
        print(f"[DEBUG] Columns: {eval_urgency_data.columns.tolist()}")
        print(f"[DEBUG] Sample input_text: {eval_urgency_data['input_text'].iloc[0][:100]}...")
        print(f"[DEBUG] Sample context length: {len(str(eval_urgency_data['context'].iloc[0]))}")
        print(f"[DEBUG] Sample generated_text (next step): {eval_urgency_data['generated_text'].iloc[0][:100]}...")
        print(f"[DEBUG] Sample generated_text length: {len(str(eval_urgency_data['generated_text'].iloc[0]))}")
        
        eval_urgency_result = evaluator.evaluate(
            data=eval_urgency_data,
            metrics=[faithfulness_urgency]
        )
        
        print(f"\n[DEBUG] Next steps evaluation result type: {type(eval_urgency_result)}")
        print(f"[DEBUG] Has to_df: {hasattr(eval_urgency_result, 'to_df')}")
        
        if eval_urgency_result and hasattr(eval_urgency_result, 'to_df'):
            urgency_metrics_df = eval_urgency_result.to_df()
            
            print(f"[DEBUG] Next steps metrics DataFrame shape: {urgency_metrics_df.shape}")
            print(f"[DEBUG] Next steps metrics DataFrame columns: {urgency_metrics_df.columns.tolist()}")
            print(f"[DEBUG] Next steps metrics DataFrame dtypes:\n{urgency_metrics_df.dtypes}")
            print(f"[DEBUG] Next steps metrics DataFrame head:\n{urgency_metrics_df.head()}")
            
            if not urgency_metrics_df.empty:
                print(f"\n[SUCCESS] Next steps evaluation complete")
                
                # Add message_ids to metrics
                urgency_metrics_df["message_id"] = valid_next_steps["message_id"].values
                
                # Merge with next steps results
                next_steps_with_metrics = next_steps_df.merge(urgency_metrics_df, on="message_id", how="left")
                
                print("\n" + "="*80)
                print("NEXT STEPS EVALUATION SUMMARY")
                print("="*80)
                
                # Find metric score column
                score_candidates = [
                    'faithfulness.llm_as_judge',
                    'value',
                    'score',
                    'metric_value'
                ]
                score_col = next((col for col in score_candidates if col in urgency_metrics_df.columns), None)

                if score_col is None:
                    numeric_metric_cols = [
                        col for col in urgency_metrics_df.columns
                        if col != "message_id" and pd.api.types.is_numeric_dtype(urgency_metrics_df[col])
                    ]
                    score_col = numeric_metric_cols[0] if numeric_metric_cols else None
                
                if score_col:
                    urgency_metrics_df["faithfulness_score"] = urgency_metrics_df[score_col]
                    next_steps_with_metrics = next_steps_df.merge(
                        urgency_metrics_df[["message_id", "faithfulness_score"]],
                        on="message_id",
                        how="left"
                    )

                    avg_score = urgency_metrics_df["faithfulness_score"].mean()
                    min_score = urgency_metrics_df["faithfulness_score"].min()
                    max_score = urgency_metrics_df["faithfulness_score"].max()
                    
                    print(f"Detected score column: {score_col}")
                    print(f"Average Next Steps Faithfulness Score: {avg_score:.2f}")
                    print(f"Min Score: {min_score:.2f}")
                    print(f"Max Score: {max_score:.2f}")
                    
                    if avg_score >= 0.8:
                        print("\nNext Steps Quality Assessment: EXCELLENT")
                    elif avg_score >= 0.6:
                        print("\nNext Steps Quality Assessment: GOOD")
                    elif avg_score >= 0.4:
                        print("\nNext Steps Quality Assessment: FAIR")
                    else:
                        print("\nNext Steps Quality Assessment: NEEDS IMPROVEMENT")
                else:
                    print("[WARNING] Could not find numeric score column in next steps evaluation")
                    next_steps_with_metrics = next_steps_df
            else:
                print("[WARNING] No metrics returned from next steps evaluation")
                next_steps_with_metrics = next_steps_df
        else:
            print("[WARNING] Next steps evaluation returned no result")
            next_steps_with_metrics = next_steps_df
            
    except Exception as e:
        print(f"[ERROR] During next steps evaluation: {str(e)}")
        import traceback
        traceback.print_exc()
        next_steps_with_metrics = next_steps_df
else:
    print("[WARNING] No valid next steps to evaluate")
    next_steps_with_metrics = next_steps_df


EVALUATING NEXT STEPS QUALITY WITH WATSONX.GOVERNANCE

Evaluating 1 high urgency next steps...

[DEBUG] Next steps data shape: (1, 3)
[DEBUG] Columns: ['input_text', 'context', 'generated_text']
[DEBUG] Sample input_text: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what co...
[DEBUG] Sample context length: 500
[DEBUG] Sample generated_text (next step): No action identified...
[DEBUG] Sample generated_text length: 20
[Warning] No region provided : Using default region as us-south

[DEBUG] Next steps evaluation result type: <class 'ibm_watsonx_gov.entities.evaluation_result.MetricsEvaluationResult'>
[DEBUG] Has to_df: True
[DEBUG] Next steps metrics DataFrame shape: (1, 1)
[DEBUG] Next steps metrics DataFrame columns: ['faithfulness.llm_as_judge']
[DEBUG] Next steps metrics DataFrame dtypes:
faithfulness.llm_as_judge    float64
dtype: object
[DEBUG] Next steps metrics DataFrame head:
   faithfulness.llm_as_judge
0                        0

## Evaluate Email Quality with Governance

In [8]:
print("="*80)
print("EVALUATING EMAIL QUALITY WITH WATSONX.GOVERNANCE")
print("="*80)

# Filter out error results
valid_results = agent_df[agent_df["risk_level"] != "Error"].copy()

if not valid_results.empty:
    # Prepare evaluation data
    eval_data = valid_results[["input_text", "context", "generated_text"]].copy()
    
    # Create faithfulness metric with configuration
    config = GenAIConfiguration(
        input_fields=["input_text"],
        context_fields=["context"],
        output_fields=["generated_text"]
    )
    
    faithfulness_metric = FaithfulnessMetric(
        llm_judge=llm_judge,
        configuration=config
    )
    
    try:
        print(f"\nEvaluating {len(eval_data)} emails...")
        print(f"\n[DEBUG] Evaluation data shape: {eval_data.shape}")
        print(f"[DEBUG] Columns: {eval_data.columns.tolist()}")
        print(f"[DEBUG] Sample input_text: {eval_data['input_text'].iloc[0][:100]}...")
        print(f"[DEBUG] Sample context length: {len(str(eval_data['context'].iloc[0]))}")
        print(f"[DEBUG] Sample generated_text length: {len(str(eval_data['generated_text'].iloc[0]))}")
        
        eval_result = evaluator.evaluate(
            data=eval_data,
            metrics=[faithfulness_metric]
        )
        
        print(f"\n[DEBUG] Evaluation result type: {type(eval_result)}")
        print(f"[DEBUG] Has to_df: {hasattr(eval_result, 'to_df')}")
        
        if eval_result and hasattr(eval_result, 'to_df'):
            metrics_df = eval_result.to_df()
            
            print(f"[DEBUG] Metrics DataFrame shape: {metrics_df.shape}")
            print(f"[DEBUG] Metrics DataFrame columns: {metrics_df.columns.tolist()}")
            print(f"[DEBUG] Metrics DataFrame dtypes:\n{metrics_df.dtypes}")
            print(f"[DEBUG] Metrics DataFrame head:\n{metrics_df.head()}")
            
            if not metrics_df.empty:
                print(f"\n[SUCCESS] Evaluation complete")
                print(f"\nMetrics DataFrame:")
                print(metrics_df)
                
                # Add message_ids to metrics
                metrics_df["message_id"] = valid_results["message_id"].values
                
                # Merge with agent results
                final_df = agent_df.merge(metrics_df, on="message_id", how="left")
                
                print("\n" + "="*80)
                print("EVALUATION SUMMARY")
                print("="*80)
                
                # Find metric score column
                score_candidates = [
                    'faithfulness.llm_as_judge',
                    'value',
                    'score',
                    'metric_value'
                ]
                score_col = next((col for col in score_candidates if col in metrics_df.columns), None)

                if score_col is None:
                    numeric_metric_cols = [
                        col for col in metrics_df.columns
                        if col != "message_id" and pd.api.types.is_numeric_dtype(metrics_df[col])
                    ]
                    score_col = numeric_metric_cols[0] if numeric_metric_cols else None
                
                if score_col:
                    metrics_df["faithfulness_score"] = metrics_df[score_col]
                    final_df = agent_df.merge(
                        metrics_df[["message_id", "faithfulness_score"]],
                        on="message_id",
                        how="left"
                    )

                    avg_score = metrics_df["faithfulness_score"].mean()
                    min_score = metrics_df["faithfulness_score"].min()
                    max_score = metrics_df["faithfulness_score"].max()
                    
                    print(f"Detected score column: {score_col}")
                    print(f"Average Faithfulness Score: {avg_score:.2f}")
                    print(f"Min Score: {min_score:.2f}")
                    print(f"Max Score: {max_score:.2f}")
                    
                    if avg_score >= 0.8:
                        print("\nOverall Assessment: EXCELLENT")
                    elif avg_score >= 0.6:
                        print("\nOverall Assessment: GOOD")
                    elif avg_score >= 0.4:
                        print("\nOverall Assessment: FAIR")
                    else:
                        print("\nOverall Assessment: NEEDS IMPROVEMENT")
                else:
                    print("[WARNING] Could not find numeric score column in results")
                    final_df = agent_df
            else:
                print("[WARNING] No metrics returned from evaluation")
                final_df = agent_df
        else:
            print("[WARNING] Evaluation returned no result")
            final_df = agent_df
            
    except Exception as e:
        print(f"[ERROR] During evaluation: {str(e)}")
        import traceback
        traceback.print_exc()
        final_df = agent_df
else:
    print("[WARNING] No valid results to evaluate")
    final_df = agent_df

EVALUATING EMAIL QUALITY WITH WATSONX.GOVERNANCE

Evaluating 1 emails...

[DEBUG] Evaluation data shape: (1, 3)
[DEBUG] Columns: ['input_text', 'context', 'generated_text']
[DEBUG] Sample input_text: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what co...
[DEBUG] Sample context length: 1003
[DEBUG] Sample generated_text length: 519

[DEBUG] Evaluation result type: <class 'ibm_watsonx_gov.entities.evaluation_result.MetricsEvaluationResult'>
[DEBUG] Has to_df: True
[DEBUG] Metrics DataFrame shape: (1, 1)
[DEBUG] Metrics DataFrame columns: ['faithfulness.llm_as_judge']
[DEBUG] Metrics DataFrame dtypes:
faithfulness.llm_as_judge    float64
dtype: object
[DEBUG] Metrics DataFrame head:
   faithfulness.llm_as_judge
0                        0.0

[SUCCESS] Evaluation complete

Metrics DataFrame:
   faithfulness.llm_as_judge
0                        0.0

EVALUATION SUMMARY
Detected score column: faithfulness.llm_as_judge
Average Faithfulness Score

## View Results

In [9]:
print("="*80)
print("FINAL RESULTS")
print("="*80)
display(final_df)

FINAL RESULTS


,message_id,input_text,generated_text,risk_level,context,faithfulness_score
0,4aacbf98-d9b8-4bed-8363-cb7dede06a3f,"I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps","Subject: Reconnecting on watsonx\n\nHi Jay,\n\nIt was great connecting with you previously on the watsonx agreement you signed. I wanted to check in on whether the sizing for the watsonx renewal has been finalized. We're eager to support your continued success with the platform.\n\nThe contract has now expired, and I'd like to help get it reinstated quickly. Scheduling a call could be helpful to address any questions you may have and ensure a smooth continuation. Would that be useful to you?\n\nRegards,\nSamantha Rodriguez",High,"{'partner_name': 'Confluent', 'contract_paths': ['docs/Confluent_IBM-1.30.2024.docx', 'docs/Confluent_IBM-1.30.2025.docx', 'docs/Confluent_IBM-5.30.2023.docx', 'docs/Confluent_IBM-7.31.2024.docx'], 'contract_results': [{'file_path': 'docs/Confluent_IBM-1.30.2024.docx', 'file_name': 'Confluent_IBM-1.30.2024.docx', 'structured_summary': {'amount': '$250,003.20', 'amount_numeric': 250003.2, 'products': ['watsonx'], 'start_date': 'Jan 31, 2024', 'term_length': '1 year(s)', 'coverage_period_start': ' | {'partner_name': 'Confluent', 'maturity_level': 'Engaged Prospect', 'sales_velocity': 'High', 'deal_blockers': [{'opportunity': 'Confluent watsonx ESA Expansion Upside', 'reason': 'Did not want to go with the expansion'}, {'opportunity': 'Cognos Usage', 'reason': 'CPO Not ready to move forward yet'}, {'opportunity': 'Confluent watsonx ESA', 'reason': 'Needed to delay renewal due to change in org structure'}], 'external_signals': {'query': 'Confluent company background news technology partnership",0.0


## View Sample Email

In [10]:
if not final_df.empty:
    sample_idx = 0
    sample = final_df.iloc[sample_idx]
    
    print("="*80)
    print("SAMPLE EMAIL")
    print("="*80)
    print(f"\nQuery: {sample['input_text']}")
    print(f"\nRisk Level: {sample['risk_level']}")
    
    # Show score if available
    if 'faithfulness_score' in sample and pd.notna(sample['faithfulness_score']):
        print(f"Faithfulness Score: {sample['faithfulness_score']:.2f}")
    else:
        for col in ['faithfulness.llm_as_judge', 'value', 'score', 'metric_value']:
            if col in sample and pd.notna(sample[col]):
                print(f"Faithfulness Score: {sample[col]:.2f}")
                break
    
    print("\n" + "-"*80)
    print("DRAFT EMAIL:")
    print("-"*80)
    print(sample["generated_text"])
else:
    print("No results to display")

SAMPLE EMAIL

Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps

Risk Level: High
Faithfulness Score: 0.00

--------------------------------------------------------------------------------
DRAFT EMAIL:
--------------------------------------------------------------------------------
Subject: Reconnecting on watsonx

Hi Jay,

It was great connecting with you previously on the watsonx agreement you signed. I wanted to check in on whether the sizing for the watsonx renewal has been finalized. We're eager to support your continued success with the platform.

The contract has now expired, and I'd like to help get it reinstated quickly. Scheduling a call could be helpful to address any questions you may have and ensure a smooth continuation. Would that be usefu

## Summary

This notebook demonstrates:
1. **Multi-agent workflow execution** - Processes seller queries through contract, research, matching, and action agents
2. **Email generation** - Creates professional outreach emails based on contract and CRM data
3. **Governance evaluation** - Uses watsonx.governance to assess email quality and faithfulness
4. **Batch processing** - Evaluates multiple queries and aggregates results

### Next Steps:
- Adjust evaluation metrics based on your quality requirements
- Integrate into production workflow for continuous quality monitoring
- Add additional metrics (e.g., ContextRelevanceMetric, HallucinationMetric)